# Scanvolver tutorial: Falcons.jlから数ringのGPU畳み込みまで

Falcons.jlで単一boresight detectorの1年分のpointingを月単位のgeneratorで生成し、指定した少数のHEALPix RINGだけを `RingScanBuffer` に蓄積します。続いて、小さなsky/beamのharmonic入力を用意してGPUで畳み込み、HDF5から `(I,Q,U)` を読み出します。

既定値は `nside=8`、`lmax=16`、1 Hz、赤道付近の3 ringです。1年分を計算するのはpointingであり、畳み込み対象は選んだringだけです。低いsampling rateと人工的なharmonic係数を使う、操作手順のための例です。

このnotebookはGPUのないローカル環境で作成しています。GPU畳み込みとその出力は未実行・未検証です。GPUがなくても入力準備とscan buffer構築を実行でき、GPU依存セルは明示的にskipします。

CPU部分は2時間分を2区間に分けた短縮実行と、一括生成との比較・月境界の確認を行いました。既定の1年分の通し実行は未確認です。


## 1. 環境

Julia 1.12と、このフォルダの `Project.toml` を使います。初回の依存解決とJuliaカーネルの登録方法は [notebook/README.md](README.md) を参照してください。

GPU部分にはNVIDIA GPUと `CUDA.functional() == true` が必要です。Jupyterを操作するPCがCPU環境でも、選択したJuliaカーネルがGPUマシンで動いていれば実行できます。既定では1 GPUを使います。

以下のセルはカーネルの作業ディレクトリがリポジトリ直下、または `notebook/` のときに動きます。


In [ ]:
using Pkg

notebook_dir = isfile(joinpath(pwd(), "tutorial.ipynb")) ?
    pwd() : joinpath(pwd(), "notebook")
isfile(joinpath(notebook_dir, "tutorial.ipynb")) ||
    error("リポジトリ直下またはnotebook/を作業ディレクトリにしてください")
Pkg.activate(notebook_dir)

using Scanvolver
import Falcons
import CUDA
using HDF5: h5open
using Printf: @printf
using Plots

gpu_available = CUDA.functional()
@info "実行環境" julia=VERSION falcons=Base.pkgversion(Falcons) gpu_available
if !gpu_available
    @info "CPUの準備セルは実行できます。GPU畳み込み・HDF5読み出しはskipします。"
end


## 2. 設定する値

`target_rings` は1-basedの連続範囲です。`2*nside` が赤道のringなので、その前後を選びます。特定の1 ringなら `r:r` を指定します。

1年は365.25日とし、12等分した区間をここでは「月」と呼びます（暦月とは異なります）。月境界はmission開始からの経過秒です。月ごとに時刻を0へ戻しません。区間は `[start, stop)` として扱い、境界sampleを二重に数えません。

`lmax` はpacked入力の上限、`lstart:lstop` は実際に計算する範囲です。この例は両者を揃えます。`κ` と `bnmax` は小規模な手順例として選んだ値であり、本番の誤差要求を保証する設定ではありません。


In [ ]:
nside = 8
lmax = 16
target_rings = (2*nside - 1):(2*nside + 1)

mission_days = 365.25
sampling_rate_hz = 1.0
chunk_count = 12
compute_precision = Float32
device_ids = [0]

cp = ConvolutionParams(
    nside_output=nside,
    lstart=0,
    lstop=lmax,
    lmax=lmax,
    numofsky=1,
    numofbeam=1,
    bnmax=2,
    κ=4,
    HWP=false,
    noise=false,
)

mission_seconds = round(Int, mission_days * 24 * 60 * 60)
@assert mission_seconds % chunk_count == 0
chunk_seconds = mission_seconds ÷ chunk_count
@assert sampling_rate_hz > 0 && isinteger(chunk_seconds * sampling_rate_hz)
@assert 1 <= first(target_rings) <= last(target_rings) <= ring_count(cp)
@assert cp.lmax >= 3  # 下の入力変換の例でl=3を使用

time_windows = [
    ((i - 1) * chunk_seconds, i * chunk_seconds)
    for i in 1:chunk_count
]
@assert first(time_windows)[1] == 0
@assert last(time_windows)[2] == mission_seconds
@assert all(time_windows[i][2] == time_windows[i+1][1] for i in 1:chunk_count-1)

show_cp(cp)
@printf("mission: %.2f days, %d chunks, %d samples\n",
        mission_days, chunk_count, round(Int, mission_seconds * sampling_rate_hz))


## 3. CPUでskyとbeamの入力を準備する

各入力はHEALPix/healpyの非負modeだけを保持するpacked形式で、形状は `(3, (lmax+1)*(lmax+2)÷2)` です。3行は `(T,E,B)`、packed indexはJuliaの1-based indexを使います。

外部データをダウンロードせずに動かせるよう、決定的な非ゼロ係数を作ります。これは人工的なskyとbeamであり、実測beamや科学解析用のCMB realizationではありません。beamには `n <= bnmax` のmodeだけを入れます。`m=0` は実数にし、`l<2` のE/Bは0にします。座標系は後述のpointingと同じ黄道座標系として扱います。

実データを使うときは `raw_alm` と `raw_blm` を置き換えてください。複数sky/beamなら、行列のvectorを渡し、`numofsky` / `numofbeam` をその個数に合わせます。


In [ ]:
packed_index(l, m, lmax) =
    m * (lmax + 1) - m * (m - 1) ÷ 2 + (l - m) + 1

function toy_coefficients(lmax; beam=false, azimuth_max=lmax)
    values = zeros(ComplexF64, 3, (lmax + 1) * (lmax + 2) ÷ 2)
    for m in 0:min(azimuth_max, lmax), l in m:lmax, row in 1:3
        row > 1 && l < 2 && continue
        phase = 0.31*l + 0.47*m + 0.23*row
        value = complex(cos(phase), m == 0 ? 0.0 : 0.4*sin(phase))
        envelope = beam ? exp(-0.008*l*(l+1)) : 1.0
        values[row, packed_index(l, m, lmax)] =
            envelope * value / (l + 1)^2
    end
    return values
end

raw_alm = toy_coefficients(cp.lmax)
raw_blm = toy_coefficients(cp.lmax; beam=true, azimuth_max=cp.bnmax)

alm = prepare_alm(cp, raw_alm; input_basis=:teb, precision=compute_precision)
blm = prepare_blm(cp, raw_blm; input_basis=:teb, precision=compute_precision)

(raw_alm_size=size(raw_alm), raw_blm_size=size(raw_blm),
 prepared_alm=alm, prepared_blm=blm)


`prepare_alm` と `prepare_blm` は `(T,E,B)` を内部の `(0,+2,-2)` に変換します。

\[
a_{+2}=-(a_E+i a_B),\qquad a_{-2}=-(a_E-i a_B).
\]

skyは負の `m` を再構成し、beamは畳み込みに必要な複素共役も適用します。呼び出し側でbeamを事前に共役にしないでください。次のセルでspin変換・負mode・beam共役を確認します。

すでにspin基底の入力なら `input_basis=:spin` を指定できます。


In [ ]:
let l = 3
    a0, ap, am = alm_slice(alm, l)
    b0, bp, bm = blm_slice(blm, l)
    nmax = calc_nmax(cp, l)
    for m in 1:min(2, cp.bnmax)
        k = packed_index(l, m, cp.lmax)
        @assert a0[1,l+m+1] ≈ Complex{compute_precision}(raw_alm[1,k])
        @assert ap[1,l+m+1] ≈ Complex{compute_precision}(-(raw_alm[2,k] + im*raw_alm[3,k]))
        @assert am[1,l+m+1] ≈ Complex{compute_precision}(-(raw_alm[2,k] - im*raw_alm[3,k]))
        @assert a0[1,l-m+1] == (-1)^m * conj(a0[1,l+m+1])
        @assert ap[1,l-m+1] == (-1)^m * conj(am[1,l+m+1])

        raw_spin = (raw_blm[1,k],
                    -(raw_blm[2,k] + im*raw_blm[3,k]),
                    -(raw_blm[2,k] - im*raw_blm[3,k]))
        for (block, value) in zip((b0, bp, bm), raw_spin)
            @assert block[nmax+m+1,1] ≈ conj(Complex{compute_precision}(value))
        end
        @assert b0[nmax-m+1,1] == (-1)^m * conj(b0[nmax+m+1,1])
        @assert bp[nmax-m+1,1] == (-1)^m * conj(bm[nmax+m+1,1])
    end
end
println("sky/beamのspin変換・負mode・共役を確認しました")


## 4. Falcons.jlのscanを定義する

単一boresight detector、HWPなし、黄道座標系 `coord="E"` を使います。以下はLiteBIRD型のscanを使った例であり、特定の実機detectorのIMOを必要としません。

`alpha` / `beta` は度、回転速度はrpm、`get_pointings` の時刻は秒、返る角度はradです。Falcons 1.0.1の実装では `get_pointings(ss, start, stop)` は `(theta, phi, psi, time)` を返し、角度配列の軸は `(sample, detector)` です。

[Falconsのpointing実装](https://github.com/yusuke-takase/Falcons.jl/blob/v1.0.1/src/function/scanning.jl) と [角度規約の説明](https://github.com/yusuke-takase/Falcons.jl/blob/v1.0.1/docs/src/Scanning.md) を参照してください。`psi` はFalconsのCOSMO/HEALPix規約をそのまま渡します。この例では追加のdetector偏光角offsetやHWP角を加えません。実機detectorへ置き換える場合は、その偏光角定義と入力skyの座標系も揃えてください。


In [ ]:
ss = Falcons.gen_ScanningStrategy(
    nside=cp.nside_output,
    duration=mission_seconds,
    sampling_rate=sampling_rate_hz,
    alpha=45.0,
    beta=50.0,
    prec_rpm=Falcons.period2rpm(192.348),
    spin_rpm=0.05,
    hwp_rpm=0.0,
    coord="E",
    start_angle=0.0,
    name=["boresight"],
)

Falcons.show_ss(ss)


## 5. 月ごとに生成して対象ringへ蓄積する

`falcons_chunk` は1区間分だけ計算し、最初のdetectorの列を取り出して `(phi, theta, psi)` の順に返します。viewを使うので、この取り出しによる角度配列のコピーはありません。

その下のgeneratorは、`build_ring_scan_buffer` が次の要素を要求したときに次の月を計算します。`collect(pointing_chunks)` や全月分の配列リストは作りません。各月の対象ring外のsampleはbufferへ保存されません。

入力chunkの参照がなくなると元の月の配列はGCの解放対象になります。対象sampleの `phi/theta/psi` は最後までbufferに残ります。また、構築中には配列の拡張やpixel分け用の一時領域も必要で、最終bufferのサイズだけがピークメモリではありません。


In [ ]:
function falcons_chunk(ss, start_s, stop_s; detector_index=1)
    theta, phi, psi, times = Falcons.get_pointings(ss, start_s, stop_s)
    @assert size(theta) == size(phi) == size(psi)
    @assert 1 <= detector_index <= size(theta, 2)
    expected = round(Int, (stop_s - start_s) * ss.sampling_rate)
    @assert size(theta, 1) == length(times) == expected
    @assert first(times) ≈ start_s
    @assert last(times) ≈ stop_s - 1/ss.sampling_rate
    return (
        @view(phi[:, detector_index]),
        @view(theta[:, detector_index]),
        @view(psi[:, detector_index]),
    )
end

pointing_chunks = (
    falcons_chunk(ss, start_s, stop_s)
    for (start_s, stop_s) in time_windows
);


In [ ]:
@time buffer = build_ring_scan_buffer(
    cp,
    pointing_chunks;
    rings=target_rings,
    progress_callback=p -> begin
        @printf("chunk %2d/%d: processed=%d, retained=%d\n",
                p.chunk_index, chunk_count, p.processed_samples, p.stored_samples)
        flush(stdout)
    end,
)

@assert buffer.processed_samples == round(Int, mission_seconds * sampling_rate_hz)
@assert buffer.first_ring == first(target_rings)
@assert buffer.last_ring == last(target_rings)
@assert sum(ring_hit_count(buffer, r) for r in target_rings) == buffer.stored_samples
buffer


ここで1年分のscan bufferが完成しています。元の1年分のTODは必要ありません。

`ring_scan_buffer_bytes` は完成後の角度配列とpixel offsetsの合計です。ringごとのhit数と、保持したpixelのhit数をCPU上で確認します。


In [ ]:
hit_summary = [(ring=r, hits=ring_hit_count(buffer, r)) for r in target_rings]
@printf("retained: %d / %d samples (%.2f%%), buffer: %.2f MiB\n",
        buffer.stored_samples, buffer.processed_samples,
        100 * buffer.stored_samples / buffer.processed_samples,
        ring_scan_buffer_bytes(buffer) / 2^20)
display(hit_summary)

buffer_pixels = buffer.first_pixel:buffer.last_pixel
pixel_hits = diff(buffer.pixel_offsets)
@assert length(pixel_hits) == length(buffer_pixels)
plot(buffer_pixels, pixel_hits;
     xlabel="HEALPix RING pixel (1-based)", ylabel="Hits",
     title="Retained pointing samples", legend=false)


## 6. GPUで畳み込み、HDF5へ出力する

以下からはGPUが必要です。GPUがないとskipし、CPUによる畳み込みへの切り替えは行いません。

`beam_batch_size=1`、`l_tile=:auto`、`max_l_tile=8` を使います。`reclaim_interval=16` は各GPUワーカーが16 ring処理するごとの明示的なメモリ回収で、この3 ringの例では周期回収は発生しません。各ringの一時配列自体は処理終了時に解放されます。

出力先は実行のたびに新しい一時ディレクトリを作るため、セルの再実行で既存ファイルを上書きしません。一時出力はJulia終了時に削除されます。残したい場合は既存の保存先ディレクトリ内の新しいパスへ変更してください。

GPUの初回実行にはコンパイルやCUDA初期化の時間が含まれます。2 GPUなら `device_ids=[0,1]` とし、GPU数以上のJulia threadでカーネルを起動してください。全オプションは [HDF5 reference](../docs/hdf5_output.md#runtime-options) にあります。


In [ ]:
output_path = nothing

if gpu_available
    @assert Threads.nthreads() >= length(device_ids)
    output_dir = mktempdir()
    output_path = convolve_to_hdf5(
        cp, buffer, alm, blm;
        output_path=joinpath(output_dir, "tutorial.h5"),
        rings=target_rings,
        devices=device_ids,
        beam_batch_size=1,
        beta_order=8,
        l_tile=:auto,
        max_l_tile=8,
        memory_fraction=0.7,
        moment_method=:gemm,
        map_precision=compute_precision,
        rcond=1e-2,
        unseen=NaN,
        reclaim_interval=16,
        sky_ids=["toy-sky"],
        beam_ids=["toy-beam"],
        metadata=Dict(
            "pointing_generator" => "Falcons.jl",
            "falcons_version" => string(Base.pkgversion(Falcons)),
            "coordinate_system" => "Ecliptic",
            "mission_seconds" => mission_seconds,
            "sampling_rate_hz" => sampling_rate_hz,
            "chunk_count" => chunk_count,
        ),
        progress_callback=r -> @printf(
            "saved ring %d: %d hits, %d active pixels, %.3f s\n",
            r.ring, r.hit_count, r.active_pixels, r.elapsed_seconds),
    )
    println("HDF5 output: ", output_path)
else
    @info "GPU convolutionをskipしました。HDF5ファイルは作成していません。"
end


## 7. 完了記録と出力mapを読む

`read_ring_hdf5` は `(sky, beam, stokes, ring_pixel)` のCPU配列を返します。Stokesの順序は `I,Q,U` です。HDF5の `map` datasetをJuliaで直接読む場合とは軸順序が異なるので、このreaderを使います。

hitのないpixelや、偏光角の被覆が不足して `rcond` 条件を満たさないpixelは `NaN` です。未処理ringと完了済みringは `rings/completed` で区別できます。


In [ ]:
ring_maps = nothing

if output_path !== nothing
    run_record = h5open(output_path, "r") do file
        (state=only(read(file["run_state"])),
         requested=read(file["rings/requested"]),
         completed=read(file["rings/completed"]))
    end
    @assert run_record.state == 1
    @assert findall(==(1), run_record.requested) == collect(target_rings)
    @assert all(==(1), run_record.completed[target_rings])

    ring_maps = Dict(r => read_ring_hdf5(output_path, r) for r in target_rings)
    for r in target_rings
        maps = ring_maps[r]
        @assert size(maps) == (cp.numofsky, cp.numofbeam, 3,
                              ring_geometry(cp, r).pixel_count)
        @printf("ring %d: size=%s, finite=%d / %d\n",
                r, string(size(maps)), count(isfinite, maps), length(maps))
    end
else
    @info "GPU出力がないため、HDF5読み出しをskipします。"
end


In [ ]:
if ring_maps !== nothing
    ring_to_plot = target_rings[cld(length(target_rings), 2)]
    geometry = ring_geometry(cp, ring_to_plot)
    maps = ring_maps[ring_to_plot]
    panels = [
        plot(rad2deg.(geometry.phi_pixels), vec(maps[1,1,s,:]);
             xlabel="Ecliptic longitude [deg]", ylabel=label * " [arbitrary units]",
             legend=false, title="ring $(ring_to_plot)")
        for (s, label) in enumerate(("I", "Q", "U"))
    ]
    plot(panels...; layout=(3,1), size=(800,700))
else
    @info "GPU出力がないため、Stokes mapのplotをskipします。"
end


## 8. 1 ring・全ringへの切り替え

最初の設定セルの `target_rings` を変え、設定以降を再実行します。

```julia
target_rings = 16:16             # 特定の1 ring
target_rings = 15:17             # 数ring
target_rings = 1:ring_count(cp)  # 全ring
```

`build_ring_scan_buffer` の `rings` を省略すると全ringを保持し、`convolve_to_hdf5` の `rings` を省略するとbuffer内の全ringを処理します。一部のringだけで作ったbufferに、後から保存していないringを指定することはできません。

全ringへ広げると最終的に保持するpointingも増えます。この1年・1 Hz・1 detectorの角度3配列だけで約0.76 GB（10進表記）となり、構築中の領域は別途必要です。sampling rateやdetector数を増やすときも同じ点に注意してください。

このnotebookの表示・assertionは入力や入出力の確認です。GPUの数値精度や物理的なbeamモデルの妥当性を検証するには、別途referenceとの比較が必要です。実行パラメータや出力仕様の詳細は [HDF5 reference](../docs/hdf5_output.md) を参照してください。
